# Label-efficient lung nodule segmentation: 3D Mamba-UNet + diffusion pretraining (LIDC-IDRI)

**Kaggle setup:** *Accelerator: GPU (T4 x1 or P100)*, *Internet: on*, and *Add Input →* **`washingtongold/lidcidri30`**,
the original TCIA LIDC-IDRI DICOM folders for about a third of the patients (41 GB).
(`justinkirby/the-cancer-imaging-archive-lidcidri` holds only the XML annotations, no CT images.)

This notebook runs the whole study:
1. It preprocesses every nodule annotated by ≥2 radiologists into 96³ cubes at 0.8 mm, keeping the 50% consensus mask and each reader's own mask.
2. **QA:** it re-creates the pylidc consensus tutorial figure for LIDC-IDRI-0078 and places our cube next to it, plus a numeric check against pylidc's native volumes.
3. It pretrains the network as a diffusion denoiser on unlabelled CT cubes.
4. It fine-tunes {UNet, Mamba-UNet, Mamba-UNet+pretraining} on 10/25/50/100% of the labelled training patients and tests them all on the same held-out patients.

Finished runs are skipped when the notebook is re-run.

In [ ]:
import time; T_START = time.time()   # the whole notebook must finish inside Kaggle's 12 h limit
!git clone -q -b rebuild https://github.com/chrisjason0505/lung-nodule-segmentation-mamba-diffusion.git repo || (cd repo && git pull -q)
%cd repo
!pip install -q "setuptools<81" pylidc==0.2.3 mambapy pydicom scikit-image
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# optional fused CUDA kernels (3-5x faster Mamba). The pure-PyTorch scan is used automatically if this fails.
INSTALL_MAMBA_SSM = True
if INSTALL_MAMBA_SSM:
    !timeout 1200 pip install -q causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -3   # give up after 20 min (source build)
!python -c "from lungseg.models.ssm import backend_name; print('Mamba backend:', backend_name())"

In [ ]:
!python -m pytest -q tests

In [ ]:
DICOM_ROOT = '/kaggle/input'          # the indexer finds the series anywhere below this
DATA = '/kaggle/working/lidc_crops'
RUNS = '/kaggle/working/runs'
QA = '/kaggle/working/qa'
!python -m lungseg.data.prepare_lidc --dicom_root "$DICOM_ROOT" --out "$DATA" --workers 4 --min_readers 2

In [ ]:
!python -m lungseg.qa --dicom_root "$DICOM_ROOT" --data "$DATA" --out "$QA" --n 60
from IPython.display import Image, Markdown, display
for f in ['pylidc_tutorial_0078.png', 'crop_grid.png']:
    display(Image(f'{QA}/{f}'))

In [ ]:
# speed / memory probe: 40 steps per architecture
!python -m lungseg.train --data "$DATA" --out /kaggle/working/probe_unet  --arch unet  --steps 40 --val_every 40 --no_test
!python -m lungseg.train --data "$DATA" --out /kaggle/working/probe_mamba --arch mamba --steps 40 --val_every 40 --no_test

In [ ]:
# Kaggle gives 2 x T4: split the grid over both GPUs.
#   GPU0: diffusion pretraining -> Mamba-UNet+DDeP at 10/25/50/100% -> Mamba-UNet 100%
#   GPU1: UNet at 10/25/50/100% -> Mamba-UNet 10/25/50%
# Every run gets an equal share of the time left before DEADLINE; the step count shrinks to fit if needed.
import subprocess, os, time
STEPS, PRE_STEPS = 5000, 5000
DEADLINE = T_START + 11.4 * 3600
base = f"python -m lungseg.experiments --data {DATA} --runs {RUNS} --preset core --steps {STEPS} --pretrain_steps {PRE_STEPS}"
now = time.time(); rem = DEADLINE - now
print(f"{rem/3600:.1f} h left for training")
jobs = {  # deadlines proportional to the number of runs in each part (pretraining counts as one run)
    0: f"{base} --configs mamba+ddep --deadline {now + rem*5/6} && {base} --configs mamba --fractions 1.0 --deadline {DEADLINE}",
    1: f"{base} --configs unet --deadline {now + rem*4/7} && {base} --configs mamba --fractions 0.1 0.25 0.5 --deadline {DEADLINE}",
}
os.makedirs(RUNS, exist_ok=True)
procs = {g: subprocess.Popen(cmd, shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)},
                             stdout=open(f"{RUNS}/gpu{g}.log", "w"), stderr=subprocess.STDOUT)
         for g, cmd in jobs.items()}
while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    for g in procs:
        print(f"--- gpu{g} ---"); print(''.join(open(f"{RUNS}/gpu{g}.log").readlines()[-3:]), flush=True)
print({g: p.returncode for g, p in procs.items()})
!python -m lungseg.experiments --data "$DATA" --runs "$RUNS" --aggregate_only

In [ ]:
display(Markdown(open(f'{RUNS}/summary.md').read()))
display(Image(f'{RUNS}/data_efficiency.png'))
best = f'{RUNS}/mamba_ddep_f1_s0/best.pt'
!python -m lungseg.visualize --data "$DATA" --ckpt "$best" --out /kaggle/working/predictions.png
display(Image('/kaggle/working/predictions.png'))